In [1]:
import sys
from pathlib import Path
import os

import pandas as pd
from textblob import TextBlob

sys.path.append(str(Path.cwd().parent))
os.chdir(Path.cwd().parent)
from src.utils.config import load_config
from src.features.build_features import add_nlp_features, encode_tabular

config = load_config()

In [2]:
df = pd.read_csv(config['data']['raw_path'])
tickets = pd.read_csv(config['data']['tickets_path'])
print(df.shape, tickets.shape)
tickets.head()

(5000, 16) (5000, 3)


,customer_id,ticket_text,category
0,CUST-00000,"Support resolved my issue quickly, very happy ...",satisfied
1,CUST-00001,I was overcharged again this month and nobody ...,billing
2,CUST-00002,"Support resolved my issue quickly, very happy ...",satisfied
3,CUST-00003,Call quality has been terrible the past few we...,service_quality
4,CUST-00004,"Support resolved my issue quickly, very happy ...",satisfied


In [3]:
sample = tickets.sample(5, random_state=1)
for _, row in sample.iterrows():
    polarity = TextBlob(row['ticket_text']).sentiment.polarity
    print(f"[{row['category']:>20}] polarity={polarity:+.2f}  ->  \"{row['ticket_text']}\"")

[     service_quality] polarity=+0.00  ->  "Technician never showed up for the scheduled appointment."
[             billing] polarity=+0.00  ->  "I was overcharged again this month and nobody explained why."
[     service_quality] polarity=-0.48  ->  "Call quality has been terrible the past few weeks."
[           satisfied] polarity=+0.50  ->  "Great experience overall, keep up the good work."
[           satisfied] polarity=+1.00  ->  "Support resolved my issue quickly, very happy with the service."


In [4]:
nlp_features = add_nlp_features(tickets)
nlp_features.head()

2026-06-23 10:30:44,815 | build_features | INFO | Extracting NLP sentiment & topic features from support tickets...


,customer_id,sentiment_polarity,frustration_score,topic_billing,topic_service,topic_leaving
0,CUST-00000,1.000000,0.000000,0,0,0
1,CUST-00001,0.000000,0.500000,1,0,0
2,CUST-00002,1.000000,0.000000,0,0,0
3,CUST-00003,-0.483333,0.741667,0,1,0
4,CUST-00004,1.000000,0.000000,0,0,0


In [5]:
merged = df.merge(nlp_features, on='customer_id', how='left')
print(merged.shape)
merged[['customer_id', 'tenure', 'monthly_charges', 'sentiment_polarity', 'frustration_score', 'topic_billing', 'topic_leaving', 'churn']].head()

(5000, 21)


,customer_id,tenure,monthly_charges,sentiment_polarity,frustration_score,topic_billing,topic_leaving,churn
0,CUST-00000,6,19.35,1.000000,0.000000,0,0,0
1,CUST-00001,56,49.27,0.000000,0.500000,1,0,0
2,CUST-00002,47,40.27,1.000000,0.000000,0,0,0
3,CUST-00003,32,67.95,-0.483333,0.741667,0,0,0
4,CUST-00004,31,45.31,1.000000,0.000000,0,0,0


In [6]:
merged.groupby('churn')['frustration_score'].mean()

churn
0    0.298726
1    0.489025
Name: frustration_score, dtype: float64

In [7]:
encoders_path = Path(config['model']['encoder_path'])
encoded, encoders = encode_tabular(merged, encoders_path)
encoded.head()

2026-06-23 10:30:46,807 | build_features | INFO | Saved categorical encoders -> models/artifacts/encoders.pkl


,customer_id,gender,senior_citizen,partner,dependents,tenure,contract,internet_service,tech_support,online_security,...,payment_method,monthly_charges,total_charges,num_support_calls,churn,sentiment_polarity,frustration_score,topic_billing,topic_service,topic_leaving
0,CUST-00000,1,0,0,0,6,0,1,0,1,...,3,19.35,125.36,1,0,1.000000,0.000000,0,0,0
1,CUST-00001,0,0,1,0,56,1,1,1,0,...,3,49.27,2761.10,0,0,0.000000,0.500000,1,0,0
2,CUST-00002,1,0,0,0,47,2,0,1,0,...,1,40.27,1690.22,1,0,1.000000,0.000000,0,0,0
3,CUST-00003,1,0,1,0,32,2,1,1,0,...,1,67.95,1967.49,2,0,-0.483333,0.741667,0,1,0
4,CUST-00004,1,0,0,0,31,0,1,0,2,...,0,45.31,1343.86,1,0,1.000000,0.000000,0,0,0


In [8]:
processed_path = Path(config['data']['processed_path'])
processed_path.parent.mkdir(parents=True, exist_ok=True)
encoded.to_csv(processed_path, index=False)
print(f'Saved -> {processed_path} ({encoded.shape[0]} rows, {encoded.shape[1]} cols)')

Saved -> data/processed/churn_features.csv (5000 rows, 21 cols)
